# H&M Personalized Fashion Recommendations - Two-Tower Retrieval (work in progress)

Exploration of a **two-tower retrieval model** built with [TensorFlow Recommenders](https://www.tensorflow.org/recommenders)
(TFRS) for the Kaggle competition
[H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations):
for every customer, predict the 12 articles they will buy in the week after the training data ends (metric: MAP@12).

- **Query tower (customer):** `customer_id` embedding + scaled `age`, projected by a dense layer.
- **Candidate tower (article):** `article_id` embedding + last observed `price`, projected by a dense layer.
- **Training objective:** TFRS `Retrieval` task (softmax over the other articles of the same batch), so that a customer's
  embedding scores the articles they bought higher than the rest.

**Status.** Data preparation, both towers and a first 10-epoch training run are implemented and executed below.
Hold-out evaluation (recall@12 / MAP@12), ScaNN indexing and the Kaggle submission are not done yet; see the last section
and the repository `README.md`.

## 1. Setup

In [8]:
!pip install -q tensorflow-recommenders

In [3]:
import os

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Embedding, IntegerLookup

# Paths: raw Kaggle CSVs in DATA_DIR, compact parquet copies in REDUCED_DIR
DATA_DIR = "data"
REDUCED_DIR = os.path.join(DATA_DIR, "reduced")

## 2. The Dataset

The competition provides (see [`data/README.md`](data/README.md) for the download commands):

* `articles.csv` - metadata for each `article_id` (product type, colour, department, description, ...)
* `customers.csv` - metadata for each `customer_id` (age, club membership, newsletter settings, postal code)
* `transactions_train.csv` - one row per purchase: date, customer, article, price and sales channel
  (repeated rows mean several units of the same article)
* `sample_submission.csv` - every customer to predict for, in the submission format
* `images/` - product images per `article_id` (not used here)

In [2]:
articles_dataset = pd.read_csv(f"{DATA_DIR}/articles.csv")
customers_dataset = pd.read_csv(f"{DATA_DIR}/customers.csv")
train_dataset = pd.read_csv(f"{DATA_DIR}/transactions_train.csv")

In [3]:
articles_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

In [4]:
customers_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371980 entries, 0 to 1371979
Data columns (total 7 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   customer_id             1371980 non-null  object 
 1   FN                      476930 non-null   float64
 2   Active                  464404 non-null   float64
 3   club_member_status      1365918 non-null  object 
 4   fashion_news_frequency  1355971 non-null  object 
 5   age                     1356119 non-null  float64
 6   postal_code             1371980 non-null  object 
dtypes: float64(3), object(4)
memory usage: 73.3+ MB


In [5]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31788324 entries, 0 to 31788323
Data columns (total 5 columns):
 #   Column            Dtype  
---  ------            -----  
 0   t_dat             object 
 1   customer_id       object 
 2   article_id        int64  
 3   price             float64
 4   sales_channel_id  int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 1.2+ GB


### 2.1 Reducing memory

With 31.8M transactions, the raw frames take well over 1 GB, most of it in the `customer_id` column,
a 64-character hexadecimal string. The cell below keeps its last 16 hex digits as an `int64` (8 bytes per value)
and stores `article_id` as `int32`. The customer `age` is min-max scaled to [0, 1] at the same time.

In [236]:
# Saving Memory
customers_dataset["customer_id"] = customers_dataset.customer_id.apply(lambda x: int(x[-16:],16) ).astype('int64')
articles_dataset["article_id"] = articles_dataset["article_id"].astype(np.int32)
train_dataset["customer_id"] = train_dataset.customer_id.apply(lambda x: int(x[-16:],16) ).astype('int64')
train_dataset["article_id"] = train_dataset["article_id"].astype(np.int32)

# Preprocessing
age = customers_dataset.age.values
age = MinMaxScaler().fit_transform(age.reshape(-1,1)).T[0]
customers_dataset["age"] = age

### 2.2 Saving as parquet

The reduced frames are saved as compressed parquet, which is much faster to reload than the CSVs.

In [7]:
# Save parquet
os.makedirs(REDUCED_DIR, exist_ok=True)
articles_dataset.to_parquet(f"{REDUCED_DIR}/articles.parquet.gzip", compression='gzip')
customers_dataset.to_parquet(f"{REDUCED_DIR}/customers.parquet.gzip", compression='gzip')
train_dataset.to_parquet(f"{REDUCED_DIR}/transactions.parquet.gzip", compression='gzip')

# Free memory before reloading the compact copies
del articles_dataset
del customers_dataset
del train_dataset
del age

In [235]:
# Read parquet
articles_dataset = pd.read_parquet(f"{REDUCED_DIR}/articles.parquet.gzip")
customers_dataset = pd.read_parquet(f"{REDUCED_DIR}/customers.parquet.gzip")
train_dataset = pd.read_parquet(f"{REDUCED_DIR}/transactions.parquet.gzip")

### 2.3 Training window

To keep iterations fast, only transactions from 2020-09-01 onward (the last three weeks of the data) are used.
All of them are used for training: there is no hold-out split yet (see *Status and next steps*).

In [237]:
# Training window: last three weeks of transactions
train_dataset = train_dataset[train_dataset.t_dat >= "2020-09-01"]

## 3. Feature Preparation

In [238]:
def merger(left, right, var, on):
    """Add column `var` from `right` to `left` by mapping on key `on` (lighter on memory than pd.merge)."""
    mapper = right[[on, var]].set_index(on).to_dict()[var]
    left[var] = left[on].map(mapper)
    return left

### 3.1 Article features

Each article gets a `price` feature: its price in the training window (the last value seen for that article),
or -1 for articles without transactions in the window.

In [239]:
# Articles
articles_dataset = merger(articles_dataset,train_dataset,"price","article_id").fillna(-1)

### 3.2 Customer features

`age` was already scaled in section 2.1; it is the only customer feature used so far.

### 3.3 Training dataset

The customer features are merged into the transactions, so each row carries everything both towers need.
Customers with an unknown age get -1.

In [241]:
# Training
train_dataset = merger(train_dataset, customers_dataset, "age", "customer_id").fillna(-1)

### 3.4 Converting to tensors

`candidates_tensor` holds every article (used by the top-k metric), `train_tensor` the shuffled transactions
in batches of 5,000.

In [242]:
# Tensors
candidates_tensor = tf.data.Dataset.from_tensor_slices(dict(articles_dataset[["article_id", "price"]]))
train_tensor = tf.data.Dataset.from_tensor_slices(dict(train_dataset[['customer_id', 'article_id', 'age', 'price']])).shuffle(100000).batch(5000).cache()

### 3.5 Vocabularies

The `IntegerLookup` layers need the vocabulary of customer and article ids to map them to contiguous indices
for the embedding tables, and the vocabulary sizes define the embedding input dimensions.

In [243]:
# Getting uniques
unique_customers_ids = customers_dataset.customer_id.unique()
unique_articles_ids = articles_dataset.article_id.unique()

unique_customers = len(unique_customers_ids)
unique_articles = len(unique_articles_ids)

The dataframes are no longer needed, so they are deleted to free memory.

In [244]:
del train_dataset
del articles_dataset
del customers_dataset

## 4. The Model

### 4.1 Customer (query) tower

`CustomerModel` maps the `customer_id` to a 200-dimensional embedding and appends the scaled `age`.
Adding side features like this is straightforward in TFRS. `QueryModel` projects the result to the 100-dimensional
space shared with the articles.

In [245]:
class CustomerModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.customer_id_model = tf.keras.Sequential()
        self.customer_id_model.add(IntegerLookup(vocabulary = unique_customers_ids,mask_token = None))
        self.customer_id_model.add(Embedding(unique_customers + 1,200))
        
    def call(self, inputs):
        reshaped_age = tf.reshape(inputs['age'],(-1,1))
        return tf.concat([self.customer_id_model(inputs["customer_id"]),
                          reshaped_age],axis=1)
    
class QueryModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding_layer = CustomerModel()
        self.dense_layers = tf.keras.Sequential()
        self.dense_layers.add(tf.keras.layers.Dense(100))

        
    def call(self, inputs):
        feature_embeddings = self.embedding_layer(inputs)
        return self.dense_layers(feature_embeddings)

### 4.2 Article (candidate) tower

Same structure for articles: a 200-dimensional `article_id` embedding plus the `price` feature,
projected to 100 dimensions by `CandidateModel`.

In [246]:
class ArticleModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.article_id_model = tf.keras.Sequential()
        self.article_id_model.add(IntegerLookup(vocabulary = unique_articles_ids,mask_token = None))
        self.article_id_model.add(Embedding(unique_articles + 1, 200))
        
    def call(self, inputs):
        reshaped_price = tf.reshape(inputs["price"],(-1,1))
        return tf.concat([self.article_id_model(inputs["article_id"]),
                          reshaped_price],axis=1)
    
class CandidateModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding_layer = ArticleModel()
        self.dense_layers = tf.keras.Sequential()
        self.dense_layers.add(tf.keras.layers.Dense(100))

        
    def call(self, inputs):
        feature_embeddings = self.embedding_layer(inputs)
        return self.dense_layers(feature_embeddings)

### 4.3 Combined model

`CombinedModel` wires both towers into a TFRS `Retrieval` task. For each batch, the loss is a softmax cross-entropy
over the query-candidate dot products, where the article actually bought is the positive and the other articles in the
batch act as negatives. The `FactorizedTopK` metric (top-k accuracy against all articles) is only computed when
`training=False`, i.e. in `evaluate`, because scoring every article at every step would slow training down a lot.

In [247]:
class CombinedModel(tfrs.models.Model):
    def __init__(self):
        super().__init__()
        self.query_model = QueryModel()
        self.candidate_model = CandidateModel()
        self.task = tfrs.tasks.Retrieval(
            metrics = tfrs.metrics.FactorizedTopK(
                candidates=candidates_tensor.batch(128).map(self.candidate_model)))
        
    def compute_loss(self, features, training = False):
        query_dict = {"customer_id":features["customer_id"],"age":features["age"]}
        query_outcome = self.query_model(query_dict)
        
        candidate_dict = {"article_id":features["article_id"],"price":features["price"]}
        candidate_outcome = self.candidate_model(candidate_dict)
        
        return self.task(query_outcome, candidate_outcome, compute_metrics = not training)

## 5. Training

Adagrad with a learning rate of 0.002, 10 epochs over the three-week training window (133 batches of 5,000 transactions).

In [263]:
model = CombinedModel()
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.002))

In [265]:
history = model.fit(
    train_tensor,
    epochs=10,
    verbose=1,
)

Epoch 1/10
133/133 [==============================] - 38s 277ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0000e+00 - loss: 42571.8490 - regularization_loss: 0.0000e+00 - total_loss: 42571.8490
Epoch 2/10
133/133 [==============================] - 36s 274ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0000e+00 - loss: 42485.2059 - regularization_loss: 0.0000e+00 - total_loss: 42485.2059
Epoch 3/10
133/133 [==============================] - 37s 277ms/step - factorized_top_k/top_1_categorical_accuracy: 0.

**Reading the log.** The loss goes down steadily (about 42.6k to 40.4k per batch over 10 epochs). The
`factorized_top_k/*` values are all zero because those metrics are *disabled during training* (see 4.3), not because
the model recommends nothing. This run does not measure retrieval quality yet: that needs the hold-out evaluation below.

## 6. Status and Next Steps

Done: data reduction, feature preparation, both towers, and a first training run.

Next:
1. **Validation.** Hold out the last week of transactions (the same horizon as the competition), train on the weeks
   before it and run `model.evaluate` on it to get top-k metrics, plus recall@12 and MAP@12 against the real purchases.
2. **Indexing.** Build an approximate nearest-neighbour index over the candidate embeddings
   (`tfrs.layers.factorized_top_k.ScaNN`, or `BruteForce` as a baseline) to retrieve the top 12 articles per customer.
3. **Submission.** Score every customer in `sample_submission.csv` (including customers with no recent purchases, which
   need a fallback such as recent best sellers), map ids back to the original formats and submit to Kaggle.
4. **Model.** More history, more features (product type, colour, department, sales channel, recency) and a ranking
   stage on top of the retrieved candidates.